# Defektr — Protocol Concept

This notebook walks through the **miner-side protocol** end to end:

1. Define the challenge spec (what the validator publishes each round)
2. Inspect an ONNX model — verify it conforms to the challenge interface
3. Build the `metadata.json` that gets uploaded alongside the model
4. Upload both files to IPFS (Pinata) — get back CIDs
5. Announce the metadata CID on-chain via `subtensor.commit()`
6. Validator side: `get_commitment()` → fetch metadata → download model

The goal is to nail down the exact contract between miners and the validator
before implementing the production modules.

In [ ]:
# Dependencies — run once
# !uv pip install onnx onnxruntime requests python-dotenv tenacity torch

In [33]:
import os
import sys
import json
import hashlib
import time
from pathlib import Path

import numpy as np
import onnx
import onnxruntime as ort

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "subnet"))

from defektr.config import NETUID, CACHE_DIR

print("Imports OK")

Imports OK


## 1. Challenge spec

In production the validator publishes this JSON to IPFS and announces the CID on-chain.
For now we define it inline — this will eventually be loaded from a file or fetched from IPFS.

In [ ]:
CHALLENGE_SPEC = {
    "challenge_id":    "defektr-001",
    "challenge_block": 1000,
    "deadline_block":  51000,
    "dataset":         "mvtec_ad",
    "category":        "bottle",
    "task":            "binary",         # "binary" |, Segmentation also possible
    "classes":         ["good", "defect"],
    "constraints": {
        "max_model_size_mb": 100,
        "min_fps":           2,
        "input_shape":       [3, 224, 224],   # C, H, W
        "input_dtype":       "float32",
        "output_shape":      [1, 1],           # [N, num_classes]
        "normalization": {
            "mean": [0.485, 0.456, 0.406],
            "std":  [0.229, 0.224, 0.225]
        },
        "output":    "prob",
        "threshold": 0.5
    },
    "metrics": { # New name?
        "accuracy_weight":   0.50,
        "speed_weight":      0.30,
        "robustness_weight": 0.20,
        "t_max_ms":          500
    }
}

C_SPEC      = CHALLENGE_SPEC["constraints"]
INPUT_SHAPE = tuple(C_SPEC["input_shape"])   # (3, 224, 224)
MODEL_PATH = "/home/luka/ws/bittensor_test/Testing_vision_models/squeezenet1.0-12.onnx"

print(json.dumps(CHALLENGE_SPEC, indent=2))

{
  "challenge_id": "defektr-001",
  "challenge_block": 1000,
  "deadline_block": 51000,
  "dataset": "mvtec_ad",
  "category": "bottle",
  "task": "binary",
  "classes": [
    "good",
    "defect"
  ],
  "constraints": {
    "max_model_size_mb": 100,
    "min_fps": 2,
    "input_shape": [
      3,
      224,
      224
    ],
    "input_dtype": "float32",
    "output_shape": [
      1,
      1
    ],
    "normalization": {
      "mean": [
        0.485,
        0.456,
        0.406
      ],
      "std": [
        0.229,
        0.224,
        0.225
      ]
    },
    "output": "prob",
    "threshold": 0.5
  },
  "scoring": {
    "accuracy_weight": 0.5,
    "speed_weight": 0.3,
    "robustness_weight": 0.2,
    "t_max_ms": 500
  }
}


## 2. Inspect the ONNX model

`inspect_onnx` is the single place that reads the ONNX graph.
All other functions work from the returned dict — the model is only loaded once.

In [35]:


def inspect_onnx(path: str) -> dict:
    """
    Load an ONNX model and return a normalised summary of its parameters.

    Returns:
        {
            "input_name":   str,
            "input_shape":  [N, C, H, W],
            "input_dtype":  str,           # normalised, e.g. "float32"
            "output_name":  str,
            "output_shape": [N, classes],  # trailing size-1 spatial dims squeezed
            "size_mb":      float,
        }
    """

    _DTYPE_ALIASES = {"float": "float32", "double": "float64"}

    model = onnx.load(path)
    onnx.checker.check_model(model)

    inp = model.graph.input[0]
    out = model.graph.output[0]

    def _shape(vi):
        return [d.dim_value for d in vi.type.tensor_type.shape.dim]

    raw_dtype = onnx.TensorProto.DataType.Name(inp.type.tensor_type.elem_type).lower()

    out_shape = _shape(out)
    # Squeeze trailing size-1 dims: [1,1000,1,1] -> [1,1000], [1,1,1,1] -> [1,1]
    squeezed  = [d for d in out_shape if d != 1] or [1]
    normalised_out = [out_shape[0]] + squeezed if out_shape[0] in (0, 1) else squeezed

    return {
        "input_name":   inp.name,
        "input_shape":  _shape(inp),
        "input_dtype":  _DTYPE_ALIASES.get(raw_dtype, raw_dtype),
        "output_name":  out.name,
        "output_shape": normalised_out,
        "size_mb":      round(Path(path).stat().st_size / 1024 / 1024, 3),
    }


onnx_info  = inspect_onnx(MODEL_PATH)
print(json.dumps(onnx_info, indent=2))

{
  "input_name": "data_0",
  "input_shape": [
    1,
    3,
    224,
    224
  ],
  "input_dtype": "float32",
  "output_name": "softmaxout_1",
  "output_shape": [
    1,
    1000
  ],
  "size_mb": 4.724
}


In [36]:
def validate_against_challenge(onnx_info: dict, spec: dict) -> list:
    """
    Compare an inspect_onnx() result against the challenge spec.
    Returns a list of violation strings — empty list means the model is valid.
    """
    c = spec["constraints"]
    violations = []

    # Input shape: C/H/W must match spec exactly.
    shape = onnx_info["input_shape"]
    if len(shape) != 4:
        violations.append(f"Expected 4D input (N,C,H,W), got {shape}")
    else:
        _, C, H, W = shape
        eC, eH, eW = c["input_shape"]
        if (C, H, W) != (eC, eH, eW):
            violations.append(f"Input spatial dims must be {c['input_shape']}, got ({C},{H},{W})")

    # Input dtype.
    if onnx_info["input_dtype"] != c["input_dtype"]:
        violations.append(f"Input dtype must be {c['input_dtype']}, got {onnx_info['input_dtype']}")

    # Output shape defines the number of output classes.
    if onnx_info["output_shape"] != c["output_shape"]:
        n_classes = c["output_shape"][-1]
        violations.append(
            f"Output shape must be {c['output_shape']} ({n_classes} class(es)), "
            f"got {onnx_info['output_shape']}"
        )

    # File size.
    if onnx_info["size_mb"] > c["max_model_size_mb"]:
        violations.append(
            f"Model {onnx_info['size_mb']} MB exceeds limit {c['max_model_size_mb']} MB"
        )

    return violations


violations = validate_against_challenge(onnx_info, CHALLENGE_SPEC)
if violations:
    print("VIOLATIONS — model will score 0:")
    for v in violations:
        print(" ", v)
else:
    print("OK — model conforms to challenge spec.")

VIOLATIONS — model will score 0:
  Output shape must be [1, 1] (1 class(es)), got [1, 1000]


## 2b. Edge deployability check

Before accepting a model the validator simulates edge hardware by:
- Forcing CPU-only execution (no GPU)
- Limiting ONNX runtime to a single thread
- Applying a `HARDWARE_FACTOR` penalty to approximate the gap between the
  validator's x86 CPU and a real edge device (e.g. Raspberry Pi 4)

**Why a penalty factor?**
Limiting threads alone doesn't simulate ARM NEON, slower clocks, or less cache.
A Raspberry Pi 4 (Cortex-A72 @ 1.8 GHz) is roughly 10–20x slower than a modern
x86 laptop single core for ONNX inference.

`HARDWARE_FACTOR` should be calibrated by running a reference model on both the
validator machine and the target device and taking the ratio of inference times.
`15.0` is a conservative estimate for RPi4 vs. a modern x86 laptop until we do that.

In [ ]:
# Calibrate this by measuring the same model on your validator machine and a real
# RPi4, then set HARDWARE_FACTOR = rpi_ms / validator_ms.
HARDWARE_FACTOR = 15.0  # conservative estimate for RPi4 vs modern x86 laptop


def check_edge_deployability(
    model_path: str,
    onnx_info: dict,
    spec: dict,
    hardware_factor: float = HARDWARE_FACTOR,
    n_warmup: int = 3,
    n_timed: int = 20,
) -> dict:
    """
    Simulate edge hardware inference and return a deployability report.

    Hardware simulation:
      - CPUExecutionProvider only  (no GPU)
      - intra_op_num_threads = 1   (single core)
      - measured time × hardware_factor → simulated edge latency

    Returns a dict:
        {
            "passed":             bool,
            "simulated_fps":      float,   # after applying hardware_factor
            "measured_fps":       float,   # raw on this machine
            "min_fps":            float,
            "simulated_avg_ms":   float,
            "measured_avg_ms":    float,
            "hardware_factor":    float,
            "peak_rss_mb":        float,
            "violations":         list[str],
        }
    """
    import resource

    c          = spec["constraints"]
    min_fps    = c["min_fps"]
    C, H, W    = c["input_shape"]
    violations = []
    measured_fps = simulated_fps = 0.0
    measured_ms  = simulated_ms  = 0.0
    peak_rss     = 0.0

    try:
        opts = ort.SessionOptions()
        opts.intra_op_num_threads     = 1
        opts.inter_op_num_threads     = 1
        opts.execution_mode           = ort.ExecutionMode.ORT_SEQUENTIAL
        opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

        session    = ort.InferenceSession(
            model_path,
            sess_options=opts,
            providers=["CPUExecutionProvider"],
        )
        input_name = onnx_info["input_name"]
        dummy      = np.random.rand(1, C, H, W).astype(np.float32)

        for _ in range(n_warmup):
            session.run(None, {input_name: dummy})

        t0 = time.perf_counter()
        for _ in range(n_timed):
            session.run(None, {input_name: dummy})
        elapsed = time.perf_counter() - t0

        measured_ms  = (elapsed / n_timed) * 1000
        measured_fps = n_timed / elapsed

        simulated_ms  = measured_ms * hardware_factor
        simulated_fps = 1000 / simulated_ms

        peak_rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024  # MB

        if simulated_fps < min_fps:
            violations.append(
                f"FPS too low: {simulated_fps:.2f} simulated fps < required {min_fps} fps "
                f"({simulated_ms:.1f} ms/image after {hardware_factor}x hardware penalty)"
            )

    except MemoryError:
        violations.append("OOM: model exceeded available memory on edge device")
    except Exception as e:
        violations.append(f"Runtime crash: {type(e).__name__}: {e}")

    return {
        "passed":           len(violations) == 0,
        "simulated_fps":    round(simulated_fps, 2),
        "measured_fps":     round(measured_fps, 2),
        "min_fps":          min_fps,
        "simulated_avg_ms": round(simulated_ms, 1),
        "measured_avg_ms":  round(measured_ms, 1),
        "hardware_factor":  hardware_factor,
        "peak_rss_mb":      round(peak_rss, 1),
        "violations":       violations,
    }


report = check_edge_deployability(MODEL_PATH, onnx_info, CHALLENGE_SPEC)
print(json.dumps(report, indent=2))
if report["passed"]:
    print(f"\nOK — {report['simulated_fps']} simulated fps on edge device (min: {report['min_fps']})")
else:
    print("\nFAIL — model is not edge deployable:")
    for v in report["violations"]:
        print(" ", v)

## 3. Build `metadata.json`

This is the file uploaded to IPFS whose CID gets committed on-chain.
The validator fetches this first, then uses `model.cid` inside it to download the actual ONNX file.

**Why only the metadata CID on-chain:**
- `subtensor.commit()` has a small payload limit
- `metadata.json` is tiny and human-readable — easy to audit
- SHA256 lets the validator skip re-downloading the model when it hasn't changed

In [13]:
def sha256_file(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

print(f"SHA256: {sha256_file(MODEL_PATH)}")
print(f"Size:   {onnx_info['size_mb']} MB")

SHA256: dec81a8684617770b3cf13fadc1d92565d1d453d23935fc6388b792d99c992bd
Size:   4.724 MB


In [14]:
def build_metadata(
    miner_hotkey: str,
    model_path: str,
    model_cid: str,
    submitted_block: int,
    spec: dict,
    architecture: str = "unknown",
) -> dict:
    """
    Build the full metadata dict for a submitted model.
    Uses inspect_onnx() internally — model is loaded once here.
    """
    info = inspect_onnx(model_path)
    c    = spec["constraints"]

    return {
        "challenge_id":    spec["challenge_id"],
        "miner_hotkey":    miner_hotkey,
        "submitted_block": submitted_block,
        "model": {
            "filename": Path(model_path).name,
            "cid":      model_cid,
            "size_mb":  info["size_mb"],
            "sha256":   sha256_file(model_path),
        },
        "inference": {
            "input_name":    info["input_name"],
            "input_shape":   info["input_shape"],
            "input_dtype":   info["input_dtype"],
            "output_name":   info["output_name"],
            "output_shape":  info["output_shape"],
            "normalization": c["normalization"],
            "output":        c["output"],
            "threshold":     c["threshold"],
        },
        "training": {
            "architecture": architecture,
            "dataset":      spec["dataset"],
            "category":     spec["category"],
        },
    }


preview = build_metadata(
    miner_hotkey="5FakeHotkeyXXXXXXXXXXXXXXXX",
    model_path=MODEL_PATH,
    model_cid="<upload model first>",
    submitted_block=0,
    spec=CHALLENGE_SPEC,
    architecture="squeezenet_demo",
)
print(json.dumps(preview, indent=2))

{
  "challenge_id": "defektr-001",
  "miner_hotkey": "5FakeHotkeyXXXXXXXXXXXXXXXX",
  "submitted_block": 0,
  "model": {
    "filename": "squeezenet1.0-12.onnx",
    "cid": "<upload model first>",
    "size_mb": 4.724,
    "sha256": "dec81a8684617770b3cf13fadc1d92565d1d453d23935fc6388b792d99c992bd"
  },
  "inference": {
    "input_name": "data_0",
    "input_shape": [
      1,
      3,
      224,
      224
    ],
    "input_dtype": "float32",
    "output_name": "softmaxout_1",
    "output_shape": [
      1,
      1000
    ],
    "normalization": {
      "mean": [
        0.485,
        0.456,
        0.406
      ],
      "std": [
        0.229,
        0.224,
        0.225
      ]
    },
    "output": "prob",
    "threshold": 0.5
  },
  "training": {
    "architecture": "squeezenet_demo",
    "dataset": "mvtec_ad",
    "category": "bottle"
  }
}


## 4. Upload to IPFS

Upload order:
1. `model.onnx` → get `model_cid`
2. Embed `model_cid` into `metadata.json`
3. `metadata.json` → get `metadata_cid`
4. Commit `metadata_cid` on-chain

Requires `PINATA_JWT` in a `.env` file at the project root.

In [21]:
import importlib.util

spec_loader = importlib.util.spec_from_file_location("ipfs", PROJECT_ROOT / "data" / "ipfs.py")
ipfs = importlib.util.module_from_spec(spec_loader)
spec_loader.loader.exec_module(ipfs)

print("ipfs module loaded")

ipfs module loaded


In [22]:
MINER_HOTKEY    = "5FakeHotkeyXXXXXXXXXXXXXXXX"  # replace with real hotkey
SUBMITTED_BLOCK = 1000                             # replace with real block number
ARCHITECTURE    = "squeezenet_demo"
CHALLENGE_ID    = CHALLENGE_SPEC["challenge_id"]

print("Uploading model.onnx ...")
model_result = ipfs.upload(
    MODEL_PATH,
    name=f"{CHALLENGE_ID}_{MINER_HOTKEY[:8]}_model.onnx",
    keyvalues={"challenge_id": CHALLENGE_ID, "type": "model", "hotkey": MINER_HOTKEY},
)
model_cid     = model_result["cid"]
model_file_id = model_result["id"]
print(f"  CID:     {model_cid}")
print(f"  File ID: {model_file_id}")

Uploading model.onnx ...
  CID:     bafybeidjvetkmex7uveb4ly4rxvztubbiuelsgyha2h577eouwsxjqg2g4
  File ID: 019d0c7f-15e9-7457-844d-157e858efe5d


In [23]:
metadata = build_metadata(
    miner_hotkey=MINER_HOTKEY,
    model_path=MODEL_PATH,
    model_cid=model_cid,
    submitted_block=SUBMITTED_BLOCK,
    spec=CHALLENGE_SPEC,
    architecture=ARCHITECTURE,
)

metadata_path = Path(MODEL_PATH).parent / "metadata.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print("Uploading metadata.json ...")
meta_result = ipfs.upload(
    str(metadata_path),
    name=f"{CHALLENGE_ID}_{MINER_HOTKEY[:8]}_metadata.json",
    keyvalues={"challenge_id": CHALLENGE_ID, "type": "metadata", "hotkey": MINER_HOTKEY},
)
metadata_cid     = meta_result["cid"]
metadata_file_id = meta_result["id"]
print(f"  CID:     {metadata_cid}")
print(f"  File ID: {metadata_file_id}")
print()
print("==> This string goes on-chain:")
print(f"    subtensor.commit(wallet, netuid, '{metadata_cid}')")

Uploading metadata.json ...
  CID:     bafkreibmjbuanc4c5dtk6kn7k7wng7v2myqjbzmgwouh6hh5kdr6aslumq
  File ID: 019d0c7f-dff9-7056-b0fb-8ee2ff8ada39

==> This string goes on-chain:
    subtensor.commit(wallet, netuid, 'bafkreibmjbuanc4c5dtk6kn7k7wng7v2myqjbzmgwouh6hh5kdr6aslumq')


## 5. Announce on-chain via `subtensor.commit()` - NOT TESTED

In [ ]:
import bittensor as bt

WALLET_PATH = str(PROJECT_ROOT.parent / "wallets")
NETWORK     = "ws://127.0.0.1:9944"

wallet    = bt.Wallet(name="miner", hotkey="default", path=WALLET_PATH)
subtensor = bt.Subtensor(network=NETWORK)

print(f"Committing: {metadata_cid}")
subtensor.commit(wallet, NETUID, metadata_cid)
print("Done.")

## 6. Validator side — fetch, verify, download

This is what the validator's `forward()` loop does for each miner UID.

In [27]:
MINER_UID = 1  # replace with real UID

fetched_metadata_cid = subtensor.get_commitment(NETUID, MINER_UID)
print(f"On-chain CID for UID {MINER_UID}: {fetched_metadata_cid}")

NameError: name 'subtensor' is not defined

In [26]:
import tempfile

def validator_fetch_model(metadata_cid: str, cache_dir: str, spec: dict) -> tuple:
    """
    Given a metadata CID:
      - Fetch metadata.json (always — it's tiny)
      - Download model.onnx only if sha256 differs from cached version
      - Validate size against the active challenge spec
    Returns (metadata_dict, local_model_path).
    """
    cache      = Path(cache_dir)
    meta_path  = cache / "metadata.json"
    model_path = cache / "model.onnx"

    ipfs.download(metadata_cid, str(meta_path))
    with open(meta_path) as f:
        meta = json.load(f)

    size_mb      = meta["model"]["size_mb"]
    model_cid    = meta["model"]["cid"]
    expected_sha = meta["model"]["sha256"]
    max_size     = spec["constraints"]["max_model_size_mb"]

    if size_mb > max_size:
        raise ValueError(f"Model too large: {size_mb} MB > {max_size} MB limit")

    if model_path.exists() and sha256_file(str(model_path)) == expected_sha:
        print("Model unchanged — using cache.")
    else:
        print(f"Downloading model (CID: {model_cid}) ...")
        ipfs.download(model_cid, str(model_path))
        actual_sha = sha256_file(str(model_path))
        if actual_sha != expected_sha:
            raise ValueError(f"SHA256 mismatch! expected {expected_sha}, got {actual_sha}")
        print("Integrity check passed.")

    return meta, str(model_path)


with tempfile.TemporaryDirectory() as tmpdir:
    meta, local_model = validator_fetch_model(fetched_metadata_cid, tmpdir, CHALLENGE_SPEC)
    print()
    print("Fetched metadata:")
    print(json.dumps(meta, indent=2))
    print(f"\nModel at: {local_model}")

NameError: name 'fetched_metadata_cid' is not defined

## 7. Quick inference smoke test

Verify the model runs and produces output in the expected range.

In [24]:
from PIL import Image

def run_inference(model_path: str, meta: dict, images: list) -> tuple:
    """
    Run ONNX inference on a list of numpy images (H, W, C) uint8.
    Handles resize and normalisation internally from meta['inference'].
    Returns (binary_predictions, inference_times_ms).
    """
    inf        = meta["inference"]
    _, C, H, W = inf["input_shape"]
    mean       = np.array(inf["normalization"]["mean"], dtype=np.float32).reshape(3, 1, 1)
    std        = np.array(inf["normalization"]["std"],  dtype=np.float32).reshape(3, 1, 1)
    threshold  = inf["threshold"]

    session    = ort.InferenceSession(model_path, providers=["CPUExecutionProvider"])
    input_name = inf["input_name"]

    preds, times = [], []
    for img_arr in images:
        pil    = Image.fromarray(img_arr).resize((W, H), Image.BILINEAR).convert("RGB")
        tensor = (np.array(pil, dtype=np.float32).transpose(2, 0, 1) / 255.0 - mean) / std
        tensor = tensor[np.newaxis]

        t0  = time.perf_counter()
        out = session.run(None, {input_name: tensor})[0]
        times.append((time.perf_counter() - t0) * 1000)

        prob = float(out.squeeze())
        preds.append(1 if prob >= threshold else 0)

    return preds, times


t_max_ms     = CHALLENGE_SPEC["scoring"]["t_max_ms"]
dummy_images = [np.random.randint(0, 255, (256, 256, 3), dtype=np.uint8) for _ in range(5)]
preds, times = run_inference(local_model, meta, dummy_images)

print("Predictions: ", preds)
print("Times (ms):  ", [f"{t:.1f}" for t in times])
print(f"Average:      {np.mean(times):.1f} ms  (budget: {t_max_ms} ms)")
print(f"Speed score:  {max(0.0, 1.0 - np.mean(times) / t_max_ms):.3f}")

NameError: name 'local_model' is not defined

## Summary — what goes where

| Artefact | Stored on | Path |
|----------|-----------|------|
| `model.onnx` | IPFS | `model_cache/{uid}/model.onnx` |
| `metadata.json` | IPFS | `model_cache/{uid}/metadata.json` |
| `metadata_cid` | On-chain (`subtensor.commit`) | Bittensor metagraph |
| `challenge_spec.json` | IPFS (published by validator) | fetched once per challenge |

**Validator cache logic (each epoch):**
1. `get_commitment(uid)` → `metadata_cid`
2. If CID changed → re-fetch `metadata.json` → check `model.sha256`
3. If SHA256 changed → re-download `model.onnx` and verify
4. Run inference on cached model → score

**What changes per challenge (in `CHALLENGE_SPEC`):**

| Field | Phase 1 | Phase 2 | Phase 3 |
|-------|---------|---------|--------|
| `category` | `bottle` | `cable` | random from 15 |
| `classes` | `[good, defect]` | `[good, defect]` | `[good, scratch, dent, ...]` |
| `output_shape` | `[1, 1]` | `[1, 1]` | `[1, N]` |
| `max_model_size_mb` | 100 | 50 | 30 |
| `min_fps` | 2 | 5 | 10 |